## Imports

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import RidgeCV, LassoCV
from sklearn.svm import LinearSVR
from sklearn.cross_decomposition import PLSRegression
from sklearn.feature_selection import f_regression

In [2]:
""" Load SP500 data """

sp_500_df = pd.read_csv(
    "../../data/raw/sp500_2010_historical.csv",
    index_col=0,      # first column is the date
    parse_dates=True  # parse dates automatically
)

# Check it loaded correctly
print(sp_500_df.head())
print(sp_500_df.tail())
print(sp_500_df.shape)

print(sp_500_df.describe(include='all'))

                  Close         High          Low         Open      Volume
Date                                                                      
2010-01-04  1132.989990  1133.869995  1116.560059  1116.560059  3991400000
2010-01-05  1136.520020  1136.630005  1129.660034  1132.660034  2491020000
2010-01-06  1137.140015  1139.189941  1133.949951  1135.709961  4972660000
2010-01-07  1141.689941  1142.459961  1131.319946  1136.270020  5270680000
2010-01-08  1144.979980  1145.390015  1136.219971  1140.520020  4389590000
                  Close         High          Low         Open      Volume
Date                                                                      
2026-06-12  7431.459961  7456.399902  7363.009766  7410.850098  4950530000
2026-06-15  7554.290039  7577.919922  7516.750000  7516.750000  5674670000
2026-06-16  7511.350098  7564.959961  7508.680176  7548.779785  5286210000
2026-06-17  7420.100098  7532.169922  7402.609863  7524.500000  5883740000
2026-06-18  7500.580078  

In [3]:
""" Load market data """

market_df = pd.read_csv(
    "../../data/raw/fixed_market_data.csv",
    index_col=0, 
    sep=';'     
)

# Check it loaded correctly
print(market_df.head())
print(market_df.tail())
print(market_df.shape)

print(market_df.describe(include='all'))

          EMP    PE    CAPE    DY     Rho   MOV     IR      RR    Y02    Y10  \
Date                                                                           
4/10/88  2,0%  12,9  14,469   3,6  -0,083    118  6,02  -1,061  7,459  8,484   
4/17/88  2,0%  12,4   13,96  3,75  -0,078    121  5,88   -0,76  7,582  8,737   
4/24/88  2,0%  12,4   13,95  3,75  -0,051    123  5,83   -0,76  7,618  8,773   
5/1/88   2,0%  12,5  14,036  3,75  -0,054  124,2  5,98   -0,76  7,728  8,873   
5/8/88   2,0%  12,3  13,761  3,82  -0,079  118,4  6,29   -0,76  7,885   8,99   

         ...       YSS          NYF     _AU   _DXY    _LCP     _TY   _OIL  \
Date     ...                                                                
4/10/88  ...  0,251622  6,896895065   450,5  89,14  2473,7   97,99  16,87   
4/17/88  ...  0,328976  2,631748517  456,25  88,31  2328,2  96,534  18,32   
4/24/88  ...  0,156475  2,631748517  449,25  88,89  2201,4   96,47  18,13   
5/1/88   ...  0,146747  2,631748517     449  89,16  21

In [4]:
""" Clean Data Types for Market Data """
object_columns = market_df.select_dtypes(include=['object']).columns

# 2. Loop through object columns and standardize datatypes
for col in object_columns:
    # Replace European commas with dots and strip invisible spaces
    market_df[col] = market_df[col].astype(str).str.replace(',', '.').str.replace('%', '').str.strip()
    
    # Force conversion to float64. Any stubborn text (like "N/A" or "-") becomes NaN
    market_df[col] = pd.to_numeric(market_df[col], errors='coerce')


# 3. Print the data types again to verify
print(market_df.dtypes)

EMP     float64
PE      float64
CAPE    float64
DY      float64
Rho     float64
MOV     float64
IR      float64
RR      float64
Y02     float64
Y10     float64
STP     float64
CF      float64
MG      float64
RV      float64
ED      float64
UN      float64
GDP     float64
M2      float64
CPI     float64
DIL     float64
YSS     float64
NYF     float64
_AU     float64
_DXY    float64
_LCP    float64
_TY     float64
_OIL    float64
_MKT    float64
_VA     float64
_GR     float64
dtype: object


## Feature Engineering

### Feature Engineering SP 500 data

In [5]:
# Log returns to normalize data
level_cols = ['Close', 'High', 'Low', 'Open']
for col in level_cols:
        if col in sp_500_df.columns:
            # ln(P_t / P_t-1)
            sp_500_df[f'{col}_LogRet'] = np.log(sp_500_df[col] / sp_500_df[col].shift(1))

sp_500_df.dropna(inplace=True)

sp_500_df.head()

,Close,High,Low,Open,Volume,Close_LogRet,High_LogRet,Low_LogRet,Open_LogRet
Date,,,,,,,,,
2010-01-05,1136.520020,1136.630005,1129.660034,1132.660034,2491020000,0.003111,0.002431,0.011664,0.014316
2010-01-06,1137.140015,1139.189941,1133.949951,1135.709961,4972660000,0.000545,0.002250,0.003790,0.002689
2010-01-07,1141.689941,1142.459961,1131.319946,1136.270020,5270680000,0.003993,0.002866,-0.002322,0.000493
2010-01-08,1144.979980,1145.390015,1136.219971,1140.520020,4389590000,0.002878,0.002561,0.004322,0.003733
2010-01-11,1146.979980,1149.739990,1142.020020,1145.959961,4255780000,0.001745,0.003791,0.005092,0.004758


In [6]:
# Daily trading range (proxy for daily volatility)
sp_500_df['Intraday_Range'] = sp_500_df['High_LogRet'] - sp_500_df['Low_LogRet']
    
# Daily trading direction
sp_500_df['Intraday_Return'] = sp_500_df['Close_LogRet'] - sp_500_df['Open_LogRet']

# Rolling 5-day and 21-day realized volatility
sp_500_df['Vol_5d'] = sp_500_df['Close_LogRet'].rolling(window=5).std()
sp_500_df['Vol_21d'] = sp_500_df['Close_LogRet'].rolling(window=21).std()
    
# Cumulative returns over the last week and month
sp_500_df['Return_5d'] = sp_500_df['Close_LogRet'].rolling(window=5).sum()
sp_500_df['Return_21d'] = sp_500_df['Close_LogRet'].rolling(window=21).sum()

# Volume day-over-day percentage change
sp_500_df['Volume_Pct_Change'] = sp_500_df['Volume'].pct_change()
    
# Volume relative to its 21-day moving average (Volume Surge)
vol_ma_21 = sp_500_df['Volume'].rolling(window=21).mean()
sp_500_df['Volume_Surge_Ratio'] = sp_500_df['Volume'] / vol_ma_21

# Lags of the close return (lag 1, 2, 3 days)
for i in [1, 2, 3]:
    sp_500_df[f'Close_LogRet_Lag{i}'] = sp_500_df['Close_LogRet'].shift(i)
    sp_500_df[f'Volume_Pct_Change_Lag{i}'] = sp_500_df['Volume_Pct_Change'].shift(i)

sp_500_df = sp_500_df.dropna()
sp_500_df.head()

,Close,High,Low,Open,Volume,Close_LogRet,High_LogRet,Low_LogRet,Open_LogRet,Intraday_Range,...,Return_5d,Return_21d,Volume_Pct_Change,Volume_Surge_Ratio,Close_LogRet_Lag1,Volume_Pct_Change_Lag1,Close_LogRet_Lag2,Volume_Pct_Change_Lag2,Close_LogRet_Lag3,Volume_Pct_Change_Lag3
Date,,,,,,,,,,,,,,,,,,,,,
2010-02-03,1097.280029,1102.719971,1093.969971,1100.670044,4285450000,-0.005489,-0.001821,0.005509,0.009696,-0.007330,...,-0.000200,-0.032026,-0.097713,0.899326,0.012890,0.164785,0.014165,-0.246680,-0.009878,-0.007254
2010-02-04,1063.109985,1097.250000,1062.780029,1097.250000,5859690000,-0.031636,-0.004973,-0.028925,-0.003112,0.023952,...,-0.019948,-0.066772,0.367345,1.189642,-0.005489,-0.097713,0.012890,0.164785,0.014165,-0.246680
2010-02-05,1066.189941,1067.130005,1044.500000,1064.119995,6438900000,0.002893,-0.027834,-0.017350,-0.030659,-0.010484,...,-0.007177,-0.064425,0.098847,1.288962,-0.031636,0.367345,-0.005489,-0.097713,0.012890,0.164785
2010-02-08,1056.739990,1071.199951,1056.510010,1065.510010,4089820000,-0.008903,0.003807,0.011433,0.001305,-0.007626,...,-0.030246,-0.077321,-0.364826,0.828036,0.002893,0.098847,-0.031636,0.367345,-0.005489,-0.097713
2010-02-09,1070.520020,1079.280029,1060.060059,1060.060059,5114260000,0.012956,0.007515,0.003355,-0.005128,0.004160,...,-0.030179,-0.067243,0.250485,1.028263,-0.008903,-0.364826,0.002893,0.098847,-0.031636,0.367345


### Feature Engineering Market Data

In [7]:
""" Growth Features """
# Drop the datasets
#market_df = market_df.drop(columns=['EMP', 'MG'], errors='ignore')

# Log returns for all level variables
asset_cols = ['_AU', '_DXY', '_LCP', '_TY', '_OIL', '_MKT', '_VA', '_GR']
for col in asset_cols:
    if col in market_df.columns:
        market_df[f'{col}_LogRet'] = np.log(market_df[col] / market_df[col].shift(1))

# Shiller CAPE Yield, the inverse of the CAPE ratio
market_df['CAPE_Yield'] = (1 / market_df['CAPE']) * 100

# Excess CAPE Yield (ECY) - Shiller's Premium over Real Rates
market_df['Excess_CAPE_Yield'] = market_df['CAPE_Yield'] - market_df['RR']
    
# Percentage change of the US Dollar Index (_DXY) over the last 13 weeks (3 months = 1 quarter)
market_df['USD_Momentum_3M'] = market_df['_DXY'].pct_change(13)

# CAPE x USD feature, adjuting for overseas earnings of the companies
market_df['Valuation_Currency_Risk'] = market_df['CAPE'] * market_df['USD_Momentum_3M']

# Weekly Value and Momentum as a composite score
market_df['Momentum_12M'] = market_df['_MKT'].pct_change(52)
market_df['Value_Rank'] = market_df['CAPE_Yield'].rank(pct=True)
market_df['Momentum_Rank'] = market_df['Momentum_12M'].rank(pct=True)
market_df['Value_Mom_Composite_Score'] = market_df['Value_Rank'] + market_df['Momentum_Rank']

# Copper and Gold as proxy of growth (productivity) and inflation expectations
# Rising ratio indicates growth expectations are outpacing market slowdown
market_df['Copper_Gold_Ratio'] = market_df['_LCP'] / market_df['_AU']
market_df['Copper_Gold_Trend_3M'] = market_df['Copper_Gold_Ratio'].pct_change(13) 
    
# Equity Risk Premium (ERP)
market_df['Earnings_Yield_Pct'] = (1 / market_df['PE']) * 100
market_df['ERP'] = market_df['Earnings_Yield_Pct'] - market_df['Y10']

# Difference in growth rate over 13 weeks (3 months, 1 quarter)
macro_yoy_cols = ['CPI', 'UN', 'GDP', 'M2']
for col in macro_yoy_cols:
    if col in market_df.columns:
        market_df[f'{col}_Acceleration_3M'] = market_df[col] - market_df[col].shift(13)

# Relative volatility in the market
risk_cols = ['MOV', 'YSS', 'NYF']
for col in risk_cols:
    if col in market_df.columns:
        # 1 Year = 52 Weeks
        rolling_mean = market_df[col].rolling(window=52).mean()
        rolling_std = market_df[col].rolling(window=52).std()
        market_df[f'{col}_ZScore_1Y'] = (market_df[col] - rolling_mean) / rolling_std



In [8]:
""" Inflation Features"""
# Earning momentum (13-week difference)
market_df['Earnings_Momentum_3M'] = market_df['ED'] - market_df['ED'].shift(13)

# Inflation Proxy (Fisher Equation): Nominal Rate (IR) - Real Rate (RR)
market_df['Inflation_Proxy'] = market_df['IR'] - market_df['RR']

# Oil Shock (13-week momentum) as proxy for supply/inflation shocks
market_df['Oil_3M_Pct'] = market_df['_OIL'].pct_change(13)

# Fed Policy Trajectory (Rate hike/cut momentum)
if 'IR' in market_df.columns:
    market_df['3M_Interest_Rate_Change'] = market_df['IR'] - market_df['IR'].shift(13)

# Steepening / Flattening Trend
if 'STP' in market_df.columns:
    market_df['Steepness_Trend_3M'] = market_df['STP'] - market_df['STP'].shift(13)

In [9]:
""" Real rates (Nominal Rate minus Inflation) Features"""
# Real Rates Shifts (1-Year)
market_df['RR_Momentum_1Y'] = market_df['RR'] - market_df['RR'].shift(52)
    
# Real Rate Volatility (13-week rolling standard deviation)
# High volatility in real rates usually forces de-leveraging in equities
market_df['RR_Volatility_13W'] = market_df['RR'].rolling(window=13).std()
    
# Real Rate Extremes (1-Year Z-Score)
rr_mean = market_df['RR'].rolling(window=52).mean()
rr_std = market_df['RR'].rolling(window=52).std()
market_df['RR_ZScore_1Y'] = (market_df['RR'] - rr_mean) / rr_std

""" Consumer Confidence Features"""
# 1 Month Consumenr Confidence change (proxy for economic shocks)
market_df['CF_Shock_1M'] = market_df['CF'] - market_df['CF'].shift(4)
    
# Consumer Sentiment Z-Score (proxy for extreme optimism/pessimism)
cf_mean = market_df['CF'].rolling(window=52).mean()
cf_std = market_df['CF'].rolling(window=52).std()
market_df['CF_ZScore_1Y'] = (market_df['CF'] - cf_mean) / cf_std

""" US Bond/Treasury Features """
# Approximated 10-Year Bond Return using Duration and Convexity
# As of late June 2026, the 10-year U.S. Treasury trades with duration of 7.9 years and a convexity 0.75.
dy = (market_df['Y10'] - market_df['Y10'].shift(1)) / 100 # Change in yield (decimal format)
market_df['Synthetic_Bond_Return'] = -7.9 * dy + 0.5 * 75 * (dy ** 2)

# Steepness, Trasury Yield Changes (1 month)
stp_change = market_df['STP'] - market_df['STP'].shift(4)
y02_change = market_df['Y02'] - market_df['Y02'].shift(4)
y10_change = market_df['Y10'] - market_df['Y10'].shift(4)
    
# Binary feature for inverted yield curve (STP < 0)
market_df['Yield_Curve_Inverted'] = (market_df['STP'] < 0).astype(int)
    
# Increase in bond prices, short-term rates decreasing (proxy for Bonds bull market)
market_df['Bull_Steepening_Flag'] = ((stp_change > 0) & (y02_change < 0)).astype(int)
    
# Long-term decrease in bond (sell off, market reversal)
market_df['Bear_Steepening_Flag'] = ((stp_change > 0) & (y10_change > 0)).astype(int)
    
# Short-term decrease in bond prices
market_df['Bear_Flattening_Flag'] = ((stp_change < 0) & (y02_change > 0)).astype(int)

In [10]:
# Drop any NaNs
print(market_df.isnull().sum().sort_values(ascending=False))
market_df = market_df.fillna(0)
display(market_df)

YSS_ZScore_1Y                53
Momentum_12M                 52
Momentum_Rank                52
Value_Mom_Composite_Score    52
RR_Momentum_1Y               52
                             ..
_OIL                          0
_MKT                          0
_VA                           0
_GR                           0
Bear_Flattening_Flag          0
Length: 71, dtype: int64


,EMP,PE,CAPE,DY,Rho,MOV,IR,RR,Y02,Y10,...,RR_Momentum_1Y,RR_Volatility_13W,RR_ZScore_1Y,CF_Shock_1M,CF_ZScore_1Y,Synthetic_Bond_Return,Yield_Curve_Inverted,Bull_Steepening_Flag,Bear_Steepening_Flag,Bear_Flattening_Flag
Date,,,,,,,,,,,,,,,,,,,,,
4/10/88,2.0,12.9,14.469,3.60,-0.083,118.0,6.020,-1.061,7.459,8.484,...,0.000,0.000000,0.000000,0.000000,0.000000,0.000000,0,0,0,0
4/17/88,2.0,12.4,13.960,3.75,-0.078,121.0,5.880,-0.760,7.582,8.737,...,0.000,0.000000,0.000000,0.000000,0.000000,-0.019747,0,0,0,0
4/24/88,2.0,12.4,13.950,3.75,-0.051,123.0,5.830,-0.760,7.618,8.773,...,0.000,0.000000,0.000000,0.000000,0.000000,-0.002839,0,0,0,0
5/1/88,2.0,12.5,14.036,3.75,-0.054,124.2,5.980,-0.760,7.728,8.873,...,0.000,0.000000,0.000000,0.000000,0.000000,-0.007862,0,0,0,0
5/8/88,2.0,12.3,13.761,3.82,-0.079,118.4,6.290,-0.760,7.885,8.990,...,0.000,0.000000,0.000000,0.011476,0.000000,-0.009192,0,0,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4/5/26,2.1,27.2,35.410,1.21,0.166,81.8,3.698,0.578,3.844,4.345,...,-0.656,0.018747,-1.169783,0.072805,1.196368,0.006901,0,0,0,1
4/12/26,2.1,28.2,36.572,1.17,0.160,72.1,3.682,0.578,3.805,4.322,...,-0.656,0.017029,-1.124954,0.000000,1.156003,0.001819,0,0,0,1
4/19/26,2.1,29.5,38.236,1.12,0.095,65.7,3.692,0.578,3.702,4.243,...,-0.670,0.015750,-1.082458,0.100952,2.681418,0.006264,0,1,0,0


## Combine Market and SP 500 data

In [11]:
""" Option 1: Merge into daily data (Forward fill)"""
# ====================================================================
# FINAL MERGE: Combine SP500 and Macro Data
# ====================================================================

# 1. Fix the UserWarning by explicitly stating the date format (Month/Day/Year)
market_df.index = pd.to_datetime(market_df.index, format='%m/%d/%y')
sp_500_df.index = pd.to_datetime(sp_500_df.index)


# Outer join brings the Sunday macro data into the daily calendar
combined_df = sp_500_df.join(market_df, how='outer')

# Forward-fill to push the Sunday macro data into Monday, Tuesday, etc.
combined_df = combined_df.ffill()

# Drop the weekends (Saturdays/Sundays) so you only have trading days left
combined_df = combined_df[combined_df.index.dayofweek < 5]

print(f"\nDaily Master DF Shape: {combined_df.shape}")

display(combined_df)


Daily Master DF Shape: (4119, 94)


,Close,High,Low,Open,Volume,Close_LogRet,High_LogRet,Low_LogRet,Open_LogRet,Intraday_Range,...,RR_Momentum_1Y,RR_Volatility_13W,RR_ZScore_1Y,CF_Shock_1M,CF_ZScore_1Y,Synthetic_Bond_Return,Yield_Curve_Inverted,Bull_Steepening_Flag,Bear_Steepening_Flag,Bear_Flattening_Flag
Date,,,,,,,,,,,,,,,,,,,,,
2010-02-03,1097.280029,1102.719971,1093.969971,1100.670044,4.285450e+09,-0.005489,-0.001821,0.005509,0.009696,-0.007330,...,0.014,0.024765,-0.308380,0.122094,1.750378,0.001423,0.0,1.0,0.0,0.0
2010-02-04,1063.109985,1097.250000,1062.780029,1097.250000,5.859690e+09,-0.031636,-0.004973,-0.028925,-0.003112,0.023952,...,0.014,0.024765,-0.308380,0.122094,1.750378,0.001423,0.0,1.0,0.0,0.0
2010-02-05,1066.189941,1067.130005,1044.500000,1064.119995,6.438900e+09,0.002893,-0.027834,-0.017350,-0.030659,-0.010484,...,0.014,0.024765,-0.308380,0.122094,1.750378,0.001423,0.0,1.0,0.0,0.0
2010-02-08,1056.739990,1071.199951,1056.510010,1065.510010,4.089820e+09,-0.008903,0.003807,0.011433,0.001305,-0.007626,...,0.014,0.026866,-0.313566,0.122094,1.675974,0.001265,0.0,0.0,0.0,0.0
2010-02-09,1070.520020,1079.280029,1060.060059,1060.060059,5.114260e+09,0.012956,0.007515,0.003355,-0.005128,0.004160,...,0.014,0.026866,-0.313566,0.122094,1.675974,0.001265,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-06-12,7431.459961,7456.399902,7363.009766,7410.850098,4.950530e+09,0.005013,0.005881,0.014457,0.016761,-0.008576,...,-0.670,0.010428,-1.003009,0.100952,2.285993,-0.006296,0.0,0.0,1.0,0.0
2026-06-15,7554.290039,7577.919922,7516.750000,7516.750000,5.674670e+09,0.016393,0.016166,0.020665,0.014189,-0.004499,...,-0.670,0.010428,-1.003009,0.100952,2.285993,-0.006296,0.0,0.0,1.0,0.0
2026-06-16,7511.350098,7564.959961,7508.680176,7548.779785,5.286210e+09,-0.005700,-0.001712,-0.001074,0.004252,-0.000638,...,-0.670,0.010428,-1.003009,0.100952,2.285993,-0.006296,0.0,0.0,1.0,0.0


In [12]:
""" Option 2: Resample to weekly data """
"""
# Since market_df reports on Sundays, resample the S&P 500 to end on Sundays 
# ('W' defaults to Sunday). This perfectly aligns the two calendars.
sp_500_weekly = sp_500_df.resample('W').last()

# Now they both land on Sundays, so an inner join works flawlessly
combined_df_weekly = sp_500_weekly.join(market_df, how='inner').dropna()

print(f"Weekly Master DF Shape: {combined_df_weekly.shape}")
display(combined_df_weekly.head(3))
"""

'\n# Since market_df reports on Sundays, resample the S&P 500 to end on Sundays \n# (\'W\' defaults to Sunday). This perfectly aligns the two calendars.\nsp_500_weekly = sp_500_df.resample(\'W\').last()\n\n# Now they both land on Sundays, so an inner join works flawlessly\ncombined_df_weekly = sp_500_weekly.join(market_df, how=\'inner\').dropna()\n\nprint(f"Weekly Master DF Shape: {combined_df_weekly.shape}")\ndisplay(combined_df_weekly.head(3))\n'

## Feature Selection Pipeline

In [ ]:
def evaluate_feature_importance(combined_df, target_horizon_days=4):
    """
    Evaluates feature importance using a consensus of Lasso, Ridge, SVR, PLS, and OLS.
    """
    df = combined_df.copy()
    
    # Define predictive target "market column (_MKT)"
    if '_MKT' not in df.columns:
        raise ValueError("Target creation failed: '_MKT' column not found in dataframe.")
        
    df[f'Target_Return_{target_horizon_days}D'] = df['_MKT'].pct_change(target_horizon_days).shift(-target_horizon_days)
    
    # Separate Features (X) and Target (y)
    y = df[f'Target_Return_{target_horizon_days}D'].copy()
    X = df.drop(columns=[f'Target_Return_{target_horizon_days}D'])
    
    # Drop forward-looking or raw price columns that shouldn't be used as predictors
    cols_to_drop = [col for col in ['Close', 'High', 'Low', 'Open', '_MKT', '_VA', '_GR', 'Volume'] if col in X.columns]
    X = X.drop(columns=cols_to_drop)

    # Handle Infinities
    # Force convert everything to pure float64 arrays
    X = X.astype(np.float64)
    y = y.astype(np.float64)
    
    # Use numpy boolean masking to force Infinities into NaNs
    X[np.isinf(X)] = np.nan
    y[np.isinf(y)] = np.nan
    
    # Drop Nans
    valid_index = X.dropna().index.intersection(y.dropna().index)
    X = X.loc[valid_index]
    y = y.loc[valid_index]
    
    # Scale features
    scaler = StandardScaler()
    X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=X.columns, index=X.index)
    y_scaled = StandardScaler().fit_transform(y.values.reshape(-1, 1)).ravel()

    importance_dict = {}

    # Run models

    # 1. LASSO REGRESSION (L1 Penalty) - The Feature Dropper
    print("Running LassoCV...")
    lasso = LassoCV(cv=5, random_state=42, max_iter=10000)
    lasso.fit(X_scaled, y_scaled)
    importance_dict['Lasso_Weight'] = np.abs(lasso.coef_)
    
    # 2. RIDGE REGRESSION (L2 Penalty) - Multicollinearity Handler
    print("Running RidgeCV...")
    ridge = RidgeCV(cv=5)
    ridge.fit(X_scaled, y_scaled)
    importance_dict['Ridge_Weight'] = np.abs(ridge.coef_)
    
    # 3. SUPPORT VECTOR REGRESSION (Linear)
    print("Running Linear SVR...")
    svr = LinearSVR(C=1.0, random_state=42, max_iter=10000)
    svr.fit(X_scaled, y_scaled)
    importance_dict['SVR_Weight'] = np.abs(svr.coef_)
    
    # 4. PARTIAL LEAST SQUARES (PLS)
    print("Running PLS Regression...")
    pls = PLSRegression(n_components=3)
    pls.fit(X_scaled, y_scaled)
    importance_dict['PLS_Weight'] = np.abs(pls.coef_).ravel()
    
    # 5. OLS / Univariate F-Test (Statistical Significance)
    print("Running OLS F-Tests...")
    f_stats, p_values = f_regression(X_scaled, y)
    importance_dict['OLS_F_Stat'] = f_stats 

    # Ranked features based on importance scores from each mode;
    results_df = pd.DataFrame(importance_dict, index=X.columns)

    rank_df = pd.DataFrame()
    for col in results_df.columns:
        rank_df[f'{col}_Rank'] = results_df[col].rank(method='min')
        
    # Calculate Average Rank (Consensus)
    results_df['Consensus_Score'] = rank_df.mean(axis=1)
    
    # Sort by the final consensus score
    results_df = results_df.sort_values(by='Consensus_Score', ascending=False)
    
    print("\n=======================================================")
    print(f"TOP 20 FEATURES FOR PREDICTING {target_horizon_days}-WEEK FORWARD RETURNS")
    print("=======================================================")
    display(results_df.head(20))
    
    # Identify useless features (Features dropped by Lasso)
    dropped_by_lasso = results_df[results_df['Lasso_Weight'] == 0].index.tolist()
    print(f"\nFeatures dropped by Lasso (Zero Coefficient): {len(dropped_by_lasso)}")
    
    return results_df


In [14]:
feature_rankings = evaluate_feature_importance(combined_df, target_horizon_days=1)

# Drop the bottom 25% of features:
threshold = feature_rankings['Consensus_Score'].quantile(0.25)
best_features = feature_rankings[feature_rankings['Consensus_Score'] > threshold].index.tolist()
combined_df[f'Target_Return_1D'] = combined_df['_MKT'].pct_change(1).shift(-1)
final_model_df = combined_df[best_features + ['Target_Return_1D']] # Keep the best features + target

Running LassoCV...
Running RidgeCV...
Running Linear SVR...
Running PLS Regression...
Running OLS F-Tests...

TOP 20 FEATURES FOR PREDICTING 1-WEEK FORWARD RETURNS


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/svm/_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


,Lasso_Weight,Ridge_Weight,SVR_Weight,PLS_Weight,OLS_F_Stat,Consensus_Score
Return_5d,0.492728,0.494558,3.215132e-05,0.243659,1014.085592,82.2
_MKT_LogRet,0.250049,0.183000,5.421261e-05,0.081197,8.258238,76.8
Return_21d,0.088666,0.174190,2.028150e-05,0.088586,188.225380,72.8
Open_LogRet,0.009383,0.151327,2.922001e-05,0.021325,204.134380,71.4
Close_LogRet_Lag1,0.018096,0.050900,3.167377e-05,0.086659,233.969054,71.4
CAPE_Yield,0.005314,0.659431,7.463734e-05,0.013013,2.311274,71.2
Synthetic_Bond_Return,0.003762,0.208668,3.193309e-05,0.015487,4.411140,69.2
Close_LogRet,0.045567,0.147185,9.582944e-06,0.027967,136.504533,65.4
Vol_5d,0.061513,0.105596,6.535417e-06,0.050845,37.203423,63.4
YSS,0.013933,0.054582,3.259013e-05,0.015367,1.766522,62.2



Features dropped completely by Lasso (Zero Coefficient): 60
